In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np, pandas as pd, json, time
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

SEED = 42
DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
DS_LABEL = {'nsl_kdd_v2': 'NSL-KDD', 'unsw_nb15_v2': 'UNSW-NB15', 'cic_ids2017_v2': 'CIC-IDS2017'}
MODELS = [f'{a}_{v}' for v in ['5class_cw', '5class_smote'] for a in ['rf', 'xgb', 'dnn']]
CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
K = 5
EPS = 1e-12
PLATT_THRESHOLD = 30        # 03e hybrid rule: isotonic at or above this many fitting samples, Platt below
ECE_N_BINS = 10

# Attack configuration
BUDGET_GRID = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.10]   # fraction of the WHOLE calibration partition
N_BATCHES = 20                       # greedy refits between 0 and the maximum budget
RULES = ['block_greedy', 'block_greedy_stealth', 'random', 'highest_pt']
STEALTH_MAX_ACC_DROP = 0.005         # adversary rejects a batch that costs more than this on their own proxy
STEALTH_MAX_ECE_RISE = 0.010
TARGETS_PER_DATASET = 2              # one fragile class, one stable class, both chosen from the attack classes
MIN_TEST_SUPPORT = 200               # a target class must have at least this many test samples

# E8: non-adversarial corollary
NOISE_RATES = [0.02, 0.05, 0.0667, 0.0753, 0.10]   # the middle two are the documented CIC-IDS2017 / CSE-CIC-IDS2018 rates
NOISE_SEEDS = 10

TABLES = Path(REPO) / 'results' / 'tables'
PREFIX = 'calib_poison'

def find_proba_file(dataset, model_name, split):
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / f'{model_name}_{split}_proba.npy'
        if p.exists():
            return p
    raise FileNotFoundError(f'{dataset}/{model_name}_{split}_proba.npy')
print('ready')


In [ ]:
def fit_one_map(x, y):
    # 03e hybrid rule. Returns (predict_fn, kind, fitted_values_on_x or None).
    if len(x) < 2 or len(np.unique(y)) < 2:
        c = float(y.mean()) if len(y) else 0.5
        return (lambda p, c=c: np.full(len(p), c)), 'constant', None
    if len(x) >= PLATT_THRESHOLD:
        iso = IsotonicRegression(out_of_bounds='clip').fit(x, y)
        return (lambda p, iso=iso: iso.predict(p)), 'isotonic', iso.predict(x)
    lr = LogisticRegression(C=1e10, solver='lbfgs').fit(x.reshape(-1, 1), y)
    return (lambda p, lr=lr: lr.predict_proba(p.reshape(-1, 1))[:, 1]), 'platt', None

def fit_ovr(P, y):
    # The standard multiclass recipe: one one-vs-rest map per class, fitted independently.
    maps, kinds, fitted = [], [], []
    for c in range(K):
        f, kind, fv = fit_one_map(P[:, c], (y == c).astype(float))
        maps.append(f); kinds.append(kind); fitted.append(fv)
    return maps, kinds, fitted

def apply_ovr(maps, P):
    # Per-class maps then renormalisation. The renormaliser is a common positive factor, so it cancels
    # from every pairwise comparison and plays no part in which class wins the argmax.
    U = np.column_stack([maps[c](P[:, c]) for c in range(K)])
    s = U.sum(1, keepdims=True)
    return np.where(s > 0, U / np.maximum(s, EPS), P), U

def ece_top(conf, corr, n_bins=ECE_N_BINS):
    b = np.clip((conf * n_bins).astype(int), 0, n_bins - 1)
    cnt = np.bincount(b, minlength=n_bins).astype(float)
    sp = np.bincount(b, weights=conf, minlength=n_bins)
    sy = np.bincount(b, weights=corr, minlength=n_bins)
    return float(np.abs(sp - sy)[cnt > 0].sum() / len(conf))

def evaluate(U, y, ref_pred=None):
    # Metrics are computed on the unnormalised per-class values U, whose argmax equals the renormalised argmax.
    pred = U.argmax(1)
    Q = U / np.maximum(U.sum(1, keepdims=True), EPS)
    conf = Q[np.arange(len(pred)), pred]
    corr = (pred == y).astype(float)
    conf_mat = np.bincount(y * K + pred, minlength=K * K).reshape(K, K).astype(float)
    tp = np.diag(conf_mat); fp = conf_mat.sum(0) - tp; fn = conf_mat.sum(1) - tp
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
        rec = np.where(tp + fn > 0, tp / (tp + fn), np.nan)
    out = {'accuracy': float(corr.mean()), 'macro_f1': float(f1.mean()), 'ece_top': ece_top(conf, corr),
           **{f'recall_{CLASS_NAMES[c]}': float(rec[c]) for c in range(K)}}
    if ref_pred is not None:
        out['tpcr'] = float((pred != ref_pred).mean())    # prediction change rate against the raw decision
    return out, pred

print('calibration and metric helpers ready')


In [ ]:
def block_ids(fitted_values):
    # Isotonic maps are step functions; samples sharing a fitted value form one pool block. Returns block id
    # per sample and the size of each sample's block. For non-isotonic maps every sample is its own block,
    # which makes the influence term vanish and the candidate unattractive, which is the correct behaviour.
    if fitted_values is None:
        return None, None
    uniq, ids = np.unique(np.round(fitted_values, 12), return_inverse=True)
    sizes = np.bincount(ids)
    return ids, sizes[ids]

def margins_and_runnerup(U, target):
    # For every sample, the margin of the target class over its best rival, and the identity of that rival.
    other = U.copy(); other[:, target] = -np.inf
    rival = other.argmax(1)
    return U[:, target] - other[np.arange(len(U)), rival], rival

def select_batch(P_cal, y_cur, maps, fitted, target, conf_class, available, take):
    """Allocate the batch across isotonic pool blocks rather than across individual samples.

    Relabelling k samples of one block changes that block's fitted value by exactly k/|block| (the 1/|block|
    step was verified exact to 1e-16 while the block does not merge). So the flips bought by a block are a
    step function of how much budget it receives, and the right move is to concentrate budget in the block
    whose margin distribution is densest just under k/|block|. A per-sample greedy cannot see this, because
    a single relabel buys almost nothing in a large block. Two levers are allocated jointly:
      target lever: relabelling a sample currently labelled target drops the target map on its target block;
      rival lever:  relabelling any sample to conf_class raises the conf_class map on its rival block.
    """
    import heapq
    U_cur = np.column_stack([maps[c](P_cal[:, c]) for c in range(K)])
    marg, rival = margins_and_runnerup(U_cur, target)
    pred_t = U_cur.argmax(1) == target
    avail = np.zeros(len(y_cur), dtype=bool); avail[available] = True

    def build(ids, sizes, cand_mask, proxy_mask):
        # per block: sorted margins of the proxies it can flip, its size, and its available candidate samples
        if ids is None:
            return {}
        blocks = {}
        for j in np.where(proxy_mask)[0]:
            blocks.setdefault(ids[j], {'marg': [], 'cand': [], 'm': sizes[j]})['marg'].append(marg[j])
        for i in np.where(cand_mask & avail)[0]:
            b = ids[i]
            if b in blocks:
                blocks[b]['cand'].append(i)
        for b in blocks:
            blocks[b]['marg'] = np.sort(np.array(blocks[b]['marg']))
        return {b: v for b, v in blocks.items() if len(v['cand'])}

    t_ids, t_sizes = block_ids(fitted[target])
    c_ids, c_sizes = block_ids(fitted[conf_class])
    lev = {'t': build(t_ids, t_sizes, y_cur == target, pred_t),
           'c': build(c_ids, c_sizes, y_cur != conf_class, pred_t & (rival == conf_class))}

    def flips(v, k):
        return int(np.searchsorted(v['marg'], k / max(v['m'], 1)))

    heap, alloc = [], {}
    for L in ('t', 'c'):
        for b, v in lev[L].items():
            alloc[(L, b)] = 0
            g = flips(v, 1)
            if g > 0:
                heapq.heappush(heap, (-g, L, b))
    chosen, used = [], set()
    while len(chosen) < take and heap:
        negg, L, b = heapq.heappop(heap)
        v = lev[L][b]
        pick = None
        while v['cand']:
            i = v['cand'].pop()
            if i not in used and avail[i]:
                pick = i; break
        if pick is None:
            continue
        chosen.append(pick); used.add(pick)
        k = alloc[(L, b)] = alloc[(L, b)] + 1
        if v['cand']:
            g2 = flips(v, k + 1) - flips(v, k)
            if g2 > 0:
                heapq.heappush(heap, (-g2, L, b))
    if len(chosen) < take:   # budget left over once every block is exhausted of useful moves
        spare = [i for i in available if i not in used]
        order = np.argsort(-P_cal[spare, target])
        chosen += [spare[j] for j in order[:take - len(chosen)]]
    return np.array(chosen[:take], dtype=int)

def pick_confusable(P_cal, y_cal, target):
    # The class the adversary routes the target into: the most frequent runner-up among true-target samples.
    m = y_cal == target
    if m.sum() == 0:
        return (target + 1) % K
    other = P_cal[m].copy(); other[:, target] = -np.inf
    return int(np.bincount(other.argmax(1), minlength=K).argmax())

def run_attack(P_cal, y_cal, P_te, y_te, target, rule, budgets, seed=SEED):
    """Relabel a budgeted number of calibration samples as the confusable class, refit the standard recipe,
    measure on the test partition. The adversary optimises on P_cal/y_cal only and never sees the test set.
    The stealth audit is computed the way a defender would compute it, on the calibration partition with the
    true labels, and is RECORDED rather than used to stop the run, so every rule reaches the same budget."""
    rng = np.random.RandomState(seed)
    n_cal = len(y_cal)
    conf_class = pick_confusable(P_cal, y_cal, target)
    raw_cal = P_cal.argmax(1)
    if rule == 'block_greedy_stealth':
        # Only relabel samples the model ALREADY calls conf_class. Moving their label to conf_class makes the
        # calibration set agree more with the model, so the defender's audit accuracy rises rather than falls.
        # This is also the most plausible real corruption: an analyst accepting the tool's suggestion.
        cand_all = np.where((y_cal != conf_class) & (raw_cal == conf_class))[0]
    else:
        cand_all = np.where(y_cal != conf_class)[0]    # any sample may be relabelled as the confusable class
    maps0, _, fitted0 = fit_ovr(P_cal, y_cal)
    _, U_te0 = apply_ovr(maps0, P_te)
    raw_pred_te = P_te.argmax(1)
    base_te, _ = evaluate(U_te0, y_te, raw_pred_te)
    _, U_cal0 = apply_ovr(maps0, P_cal)
    base_cal, _ = evaluate(U_cal0, y_cal)

    n_max = min(int(round(max(budgets) * n_cal)), len(cand_all))
    checkpoints = sorted({min(int(round(b * n_cal)), len(cand_all)) for b in budgets})
    y_cur = y_cal.copy()
    poisoned = np.zeros(n_cal, dtype=bool)
    maps, fitted = maps0, fitted0
    batch = max(1, int(np.ceil(n_max / N_BATCHES)))
    rows, next_cp = [], 0
    n_target_in_calib = int((y_cal == target).sum())

    while poisoned.sum() < n_max:
        remaining = cand_all[~poisoned[cand_all]]
        if len(remaining) == 0:
            break
        take = min(batch, n_max - int(poisoned.sum()), len(remaining))
        if rule == 'random':
            chosen = rng.choice(remaining, size=take, replace=False)
        elif rule == 'highest_pt':
            chosen = remaining[np.argsort(-P_cal[remaining, target])[:take]]
        elif rule.startswith('block_greedy'):
            chosen = select_batch(P_cal, y_cur, maps, fitted, target, conf_class, remaining, take)
        else:
            raise ValueError(rule)
        y_cur[chosen] = conf_class
        poisoned[chosen] = True
        maps, _, fitted = fit_ovr(P_cal, y_cur)
        while next_cp < len(checkpoints) and poisoned.sum() >= checkpoints[next_cp]:
            _, U_te = apply_ovr(maps, P_te)
            te, pred_te = evaluate(U_te, y_te, raw_pred_te)
            _, U_cal = apply_ovr(maps, P_cal)
            aud, _ = evaluate(U_cal, y_cal)          # the defender's audit, on true calibration labels
            routed = {CLASS_NAMES[c]: float((pred_te[(y_te == target)] == c).mean()) for c in range(K)}
            rows.append({'target': CLASS_NAMES[target], 'confusable_class': CLASS_NAMES[conf_class], 'rule': rule,
                         'n_poisoned': int(poisoned.sum()), 'budget_of_calib': poisoned.sum() / n_cal,
                         'n_target_in_calib': n_target_in_calib,
                         'frac_poisoned_from_target_class': float((y_cal[poisoned] == target).mean()) if poisoned.any() else 0.0,
                         'audit_accuracy': aud['accuracy'], 'audit_ece_top': aud['ece_top'],
                         'audit_accuracy_delta': aud['accuracy'] - base_cal['accuracy'],
                         'audit_ece_delta': aud['ece_top'] - base_cal['ece_top'],
                         'stealth_ok': bool(aud['accuracy'] >= base_cal['accuracy'] - STEALTH_MAX_ACC_DROP
                                            and aud['ece_top'] <= base_cal['ece_top'] + STEALTH_MAX_ECE_RISE),
                         **{f'test_{k}': v for k, v in te.items()},
                         **{f'routed_to_{k}': v for k, v in routed.items()},
                         'baseline_recall_target': base_te[f'recall_{CLASS_NAMES[target]}'],
                         'baseline_accuracy': base_te['accuracy'], 'baseline_macro_f1': base_te['macro_f1'],
                         'baseline_ece_top': base_te['ece_top'], 'baseline_tpcr': base_te['tpcr']})
            next_cp += 1
    return rows, base_te, conf_class, len(cand_all)
print('attack engine ready')


In [ ]:
t0 = time.time()
attack_rows, target_rows = [], []
for ds in DATASETS:
    proc = Path(REPO) / 'data' / 'processed' / ds
    y_cal = np.load(proc / 'y_calib_5class.npy'); y_te = np.load(proc / 'y_test_5class.npy')
    te_support = np.bincount(y_te, minlength=K)
    print(f'\n=== {ds}: calib={len(y_cal)} test={len(y_te)} test class support={te_support.tolist()} ===')
    for model in MODELS:
        P_cal = np.load(find_proba_file(ds, model, 'calib')).astype(np.float64)
        P_te = np.load(find_proba_file(ds, model, 'test')).astype(np.float64)
        P_cal /= np.maximum(P_cal.sum(1, keepdims=True), EPS); P_te /= np.maximum(P_te.sum(1, keepdims=True), EPS)
        # Choose targets from the ATTACK classes only: the one the clean recipe already damages most (fragile)
        # and the one it leaves most intact (stable). Suppressing Normal is a different attack and is excluded.
        maps0, _, _ = fit_ovr(P_cal, y_cal)
        _, U_te0 = apply_ovr(maps0, P_te)
        clean, _ = evaluate(U_te0, y_te, P_te.argmax(1))
        raw, _ = evaluate(P_te, y_te)
        elig = [c for c in range(1, K) if te_support[c] >= MIN_TEST_SUPPORT and np.sum(y_cal == c) >= 50]
        if not elig:
            print(f'  {model}: no eligible target class'); continue
        damage = {c: raw[f'recall_{CLASS_NAMES[c]}'] - clean[f'recall_{CLASS_NAMES[c]}'] for c in elig}
        fragile = max(damage, key=damage.get)
        stable = max([c for c in elig if c != fragile], key=lambda c: clean[f'recall_{CLASS_NAMES[c]}']) if len(elig) > 1 else fragile
        targets = [(fragile, 'fragile'), (stable, 'stable')][:TARGETS_PER_DATASET]
        target_rows.append({'dataset': ds, 'model': model, 'fragile_target': CLASS_NAMES[fragile],
                            'stable_target': CLASS_NAMES[stable], 'clean_recipe_damage_fragile': damage[fragile],
                            **{f'clean_recall_{CLASS_NAMES[c]}': clean[f'recall_{CLASS_NAMES[c]}'] for c in range(K)},
                            **{f'raw_recall_{CLASS_NAMES[c]}': raw[f'recall_{CLASS_NAMES[c]}'] for c in range(K)}})
        for target, kind in targets:
            for rule in RULES:
                rows, base_te, conf_class, n_cand = run_attack(P_cal, y_cal, P_te, y_te, target, rule, BUDGET_GRID)
                for r in rows:
                    r.update({'dataset': ds, 'model': model, 'target_kind': kind, 'n_target_in_calib': n_cand})
                attack_rows += rows
                if rows:
                    last = rows[-1]
                    print(f'  {model:18s} {kind:7s} target={CLASS_NAMES[target]:6s} -> {CLASS_NAMES[conf_class]:6s} {rule:17s} '
                          f'budget={last["budget_of_calib"]*100:5.2f}% recall {base_te[f"recall_{CLASS_NAMES[target]}"]:.3f} -> {last[f"test_recall_{CLASS_NAMES[target]}"]:.3f}  '
                          f'acc {base_te["accuracy"]:.3f} -> {last["test_accuracy"]:.3f}  ECE {base_te["ece_top"]:.4f} -> {last["test_ece_top"]:.4f}  TPCR {last["test_tpcr"]*100:.1f}%')
df_attack = pd.DataFrame(attack_rows); df_attack.to_csv(TABLES / f'{PREFIX}_budget_curves.csv', index=False)
pd.DataFrame(target_rows).to_csv(TABLES / f'{PREFIX}_targets.csv', index=False)
print(f'\n{len(df_attack)} rows; {(time.time() - t0) / 60:.1f} min')


In [ ]:
t0 = time.time()
noise_rows = []
for ds in DATASETS:
    proc = Path(REPO) / 'data' / 'processed' / ds
    y_cal = np.load(proc / 'y_calib_5class.npy'); y_te = np.load(proc / 'y_test_5class.npy')
    for model in MODELS:
        P_cal = np.load(find_proba_file(ds, model, 'calib')).astype(np.float64)
        P_te = np.load(find_proba_file(ds, model, 'test')).astype(np.float64)
        P_cal /= np.maximum(P_cal.sum(1, keepdims=True), EPS); P_te /= np.maximum(P_te.sum(1, keepdims=True), EPS)
        maps0, _, _ = fit_ovr(P_cal, y_cal); _, U0 = apply_ovr(maps0, P_te)
        base, _ = evaluate(U0, y_te, P_te.argmax(1))
        for rate in NOISE_RATES:
            for seed in range(NOISE_SEEDS):
                rng = np.random.RandomState(1000 * seed + int(rate * 10000))
                y_n = y_cal.copy()
                idx = rng.choice(len(y_cal), size=int(round(rate * len(y_cal))), replace=False)
                y_n[idx] = (y_cal[idx] + rng.randint(1, K, size=len(idx))) % K   # uniform flip to a different class
                maps, _, _ = fit_ovr(P_cal, y_n); _, U = apply_ovr(maps, P_te)
                m, _ = evaluate(U, y_te, P_te.argmax(1))
                row = {'dataset': ds, 'model': model, 'noise_rate': rate, 'seed': seed,
                       **{f'test_{k}': v for k, v in m.items()}}
                for c in range(1, K):
                    b = base[f'recall_{CLASS_NAMES[c]}']
                    row[f'recall_drop_{CLASS_NAMES[c]}'] = (b - m[f'recall_{CLASS_NAMES[c]}']) if np.isfinite(b) else np.nan
                    row[f'halved_{CLASS_NAMES[c]}'] = bool(np.isfinite(b) and b > 0.1 and m[f'recall_{CLASS_NAMES[c]}'] < 0.5 * b)
                noise_rows.append(row)
        print(f'  {ds:15s} {model:18s} done')
df_noise = pd.DataFrame(noise_rows); df_noise.to_csv(TABLES / f'{PREFIX}_natural_noise.csv', index=False)
halve_cols = [f'halved_{CLASS_NAMES[c]}' for c in range(1, K)]
df_noise['any_class_halved'] = df_noise[halve_cols].any(axis=1)
summ_noise = df_noise.groupby(['dataset', 'noise_rate']).agg(
    runs=('seed', 'size'), any_class_halved=('any_class_halved', 'mean'), **{c: (c, 'mean') for c in halve_cols},
    accuracy=('test_accuracy', 'mean'), macro_f1=('test_macro_f1', 'mean'), ece=('test_ece_top', 'mean'), tpcr=('test_tpcr', 'mean')).reset_index()
summ_noise.to_csv(TABLES / f'{PREFIX}_natural_noise_summary.csv', index=False)
pd.set_option('display.width', 250)
print(summ_noise.round(4).to_string(index=False))
print(f'{(time.time() - t0) / 60:.1f} min')


In [ ]:
def min_budget_for(df, frac):
    # smallest budget at which the target's test recall falls to <= frac of its clean value
    out = []
    for (ds, model, kind, rule, target), g in df.groupby(['dataset', 'model', 'target_kind', 'rule', 'target']):
        g = g.sort_values('n_poisoned'); base = g.baseline_recall_target.iloc[0]
        col = f'test_recall_{target}'
        hit = g[g[col] <= frac * base]
        out.append({'dataset': ds, 'model': model, 'target_kind': kind, 'rule': rule, 'target': target,
                    'baseline_recall': base, 'final_recall': g[col].iloc[-1],
                    'min_budget': float(hit.budget_of_calib.iloc[0]) if len(hit) else np.nan,
                    'frac_from_target_class_at_min': float(hit.frac_poisoned_from_target_class.iloc[0]) if len(hit) else np.nan,
                    'max_budget_tested': float(g.budget_of_calib.iloc[-1]),
                    'acc_delta': float(g.test_accuracy.iloc[-1] - g.baseline_accuracy.iloc[0]),
                    'macro_f1_delta': float(g.test_macro_f1.iloc[-1] - g.baseline_macro_f1.iloc[0]),
                    'ece_delta': float(g.test_ece_top.iloc[-1] - g.baseline_ece_top.iloc[0]),
                    'tpcr': float(g.test_tpcr.iloc[-1]),
                    'stealth_ok_at_min_budget': bool(hit.stealth_ok.iloc[0]) if len(hit) else False,
                    'audit_acc_delta': float(g.audit_accuracy_delta.iloc[-1]), 'audit_ece_delta': float(g.audit_ece_delta.iloc[-1])})
    return pd.DataFrame(out)

half = min_budget_for(df_attack, 0.5); half.to_csv(TABLES / f'{PREFIX}_min_budget_half.csv', index=False)
erase = min_budget_for(df_attack, 0.1); erase.to_csv(TABLES / f'{PREFIX}_min_budget_erase.csv', index=False)

print('=== E1: smallest budget (share of the calibration partition) that halves the target class recall ===')
print(half.groupby(['dataset', 'target_kind', 'rule']).agg(
    models_halved=('min_budget', lambda s: int(s.notna().sum())), n_models=('min_budget', 'size'),
    median_budget=('min_budget', 'median'), median_final_recall=('final_recall', 'median'),
    median_baseline=('baseline_recall', 'median'), median_acc_delta=('acc_delta', 'median'),
    median_ece_delta=('ece_delta', 'median'), median_tpcr=('tpcr', 'median')).round(4).to_string())
print()
print('=== E2: selection rule (median over models) ===')
print(half.pivot_table(index=['dataset', 'target_kind'], columns='rule', values='min_budget', aggfunc='median').round(4).to_string())
print(half.pivot_table(index=['dataset', 'target_kind'], columns='rule', values='final_recall', aggfunc='median').round(3).to_string())
print()
print('=== damage against stealth: audit deltas the defender would see at the maximum budget ===')
print(half.pivot_table(index='rule', values=['final_recall', 'audit_acc_delta', 'audit_ece_delta', 'acc_delta', 'ece_delta'], aggfunc='median').round(4).to_string())
print()
print('=== stealthy attacks only (defender audit within tolerance at the halving budget) ===')
st = half[half.stealth_ok_at_min_budget]
print(st.groupby(['dataset', 'rule']).agg(models=('min_budget', 'size'), median_budget=('min_budget', 'median'),
      median_final_recall=('final_recall', 'median')).round(4).to_string() if len(st) else '  none: no rule halved a class while staying inside the audit tolerance')
print()
print('=== E3: fragile against stable targets ===')
print(half.groupby(['dataset', 'target_kind']).agg(median_baseline=('baseline_recall', 'median'),
      median_final=('final_recall', 'median'), models_halved=('min_budget', lambda s: int(s.notna().sum()))).round(3).to_string())
print()
print('=== E1 erase (recall below 10% of clean) ===')
print(erase.groupby(['dataset', 'target_kind', 'rule']).agg(models_erased=('min_budget', lambda s: int(s.notna().sum())),
      median_budget=('min_budget', 'median')).round(4).to_string())

g = half[half.rule == 'block_greedy']
crit = {'timestamp': datetime.now().isoformat(), 'notebook': '17_calibration_poisoning.ipynb',
        'budget_grid': BUDGET_GRID, 'rules': RULES, 'stealth_max_acc_drop': STEALTH_MAX_ACC_DROP,
        'stealth_max_ece_rise': STEALTH_MAX_ECE_RISE, 'noise_rates': NOISE_RATES, 'noise_seeds': NOISE_SEEDS,
        'KILL_1_practical_budget': {'rule': 'fails if the median budget to halve is above 10% on all three datasets',
            'median_budget_by_dataset': g.groupby('dataset').min_budget.median().round(4).to_dict(),
            'models_halved_by_dataset': g.groupby('dataset').min_budget.apply(lambda s: int(s.notna().sum())).to_dict()},
        'KILL_2_algorithm_matters': {'rule': 'fails if block_greedy is not clearly better than random AND highest_pt',
            'median_budget_by_rule': half.groupby('rule').min_budget.median().round(4).to_dict(),
            'median_final_recall_by_rule': half.groupby('rule').final_recall.median().round(4).to_dict()},
        'KILL_3_stable_targets': {'rule': 'weak if only already-fragile classes can be suppressed',
            'models_halved_fragile': int(g[g.target_kind == 'fragile'].min_budget.notna().sum()),
            'models_halved_stable': int(g[g.target_kind == 'stable'].min_budget.notna().sum()),
            'n_per_kind': int((g.target_kind == 'fragile').sum())},
        'STEALTH_DEFENDER_AUDIT': {'median_audit_acc_delta': float(g.audit_acc_delta.median()),
                    'median_audit_ece_delta': float(g.audit_ece_delta.median()),
                    'frac_min_budget_points_stealthy': float(g.stealth_ok_at_min_budget.mean())},
        'STEALTH': {'median_accuracy_delta': float(g.acc_delta.median()), 'median_macro_f1_delta': float(g.macro_f1_delta.median()),
                    'median_ece_delta': float(g.ece_delta.median()), 'median_tpcr': float(g.tpcr.median())}}
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump(crit, f, indent=2, default=float)
print('\n=== KILL CRITERIA ===')
print(json.dumps(crit['KILL_1_practical_budget'], indent=1, default=float))
print(json.dumps(crit['KILL_2_algorithm_matters'], indent=1, default=float))
print(json.dumps(crit['KILL_3_stable_targets'], indent=1, default=float))
print(json.dumps(crit['STEALTH'], indent=1, default=float))
print(json.dumps(crit['STEALTH_DEFENDER_AUDIT'], indent=1, default=float))


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"
import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '17_calibration_poisoning.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')
!git add notebooks/17_calibration_poisoning.ipynb results/tables/calib_poison_*.csv results/tables/calib_poison_summary.json
!git status --short | head -20
!git commit -m "Notebook 17: calibration-set label poisoning against the one-vs-rest plus renormalisation recipe. Closed-form PAVA block influence, greedy batch selection under a stealth constraint, budget sweep against random and highest-score selection, fragile and stable targets, natural label-noise corollary"
!git push origin main
!git log --oneline -2
